In [2]:
## Python 3.12.7
!pip install pandas pyarrow

In [ ]:
import pandas as pd
import os

# ==========================================
# 1. Configure file paths (Please replace with your actual local paths)
# ==========================================
ED_STAYS_PATH = '../data/edstays.csv.gz'
CXR_META_PATH = '../data/mimic-cxr-2.0.0-metadata.csv.gz' ## Extracted CXR metadata
CXR_LABEL_PATH = '../data/mimic-cxr-2.0.0-chexpert.csv.gz' ## Extracted CXR labels
OUTPUT_PATH = '../data/ed_patients_first_ap_cxr.csv' ## Saved CXR data

print("Loading data...")

# ==========================================
# 2. Extract unique patient IDs from MIMIC-IV-ED
# ==========================================
# Only read the subject_id column to save memory
ed_df = pd.read_csv(ED_STAYS_PATH, usecols=['subject_id'])
ed_patients = set(ed_df['subject_id'].unique())
print(f"Found {len(ed_patients)} unique patients in the ED dataset.")

# ==========================================
# 3. Process MIMIC-CXR metadata
# ==========================================
# Load the required core columns, including time information for sorting
meta_cols = ['subject_id', 'study_id', 'dicom_id', 'ViewPosition', 'StudyDate', 'StudyTime']
cxr_meta = pd.read_csv(CXR_META_PATH, usecols=meta_cols)

# Filter 1: Keep only patients present in the Emergency Department (ED) database
filtered_meta = cxr_meta[cxr_meta['subject_id'].isin(ed_patients)]

# Filter 2: Keep only AP (Anteroposterior) view
filtered_meta = filtered_meta[filtered_meta['ViewPosition'] == 'AP']

# ==========================================
# 4. Sort and extract the "first" image for each patient
# ==========================================
# Sort in ascending order by patient ID, study date, and study time
filtered_meta = filtered_meta.sort_values(by=['subject_id', 'StudyDate', 'StudyTime'])

# Drop duplicates based on subject_id, keep='first' ensures only the earliest record is retained
first_cxr_df = filtered_meta.drop_duplicates(subset=['subject_id'], keep='first')
print(f"After filtering and deduplication, {len(first_cxr_df)} unique first AP view images remain.")

# ==========================================
# 5. Merge diagnostic labels (CheXpert)
# ==========================================
# Load CheXpert labels (contains 14 pathologies including Pneumonia)
cxr_labels = pd.read_csv(CXR_LABEL_PATH)

# Left join image metadata and labels on subject_id and study_id
final_df = pd.merge(first_cxr_df, cxr_labels, on=['subject_id', 'study_id'], how='left')

# ==========================================
# 6. Save as CSV file
# ==========================================
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Processing complete! Results saved to: {OUTPUT_PATH}")

# Print file information
file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Generated CSV file size: {file_size_mb:.2f} MB")

正在加载数据...
在 ED 数据集中找到 205504 名独立患者。
过滤并去重后，剩余 31719 张独立的首次正位片。
处理完成！结果已保存至: ../data/ed_patients_first_ap_cxr.csv
生成的 CSV 文件大小: 3.29 MB


In [ ]:
# Read the previously generated CSV containing 31.7k records
df = pd.read_csv('../data/ed_patients_first_ap_cxr.csv')

# 1. Strict filtering: Keep only records explicitly labeled as 1.0 (Pneumonia) or 0.0 (No Pneumonia)
# Automatically drops NaN and -1.0 (uncertain)
filtered_df = df[df['Pneumonia'].isin([1.0, 0.0])]

# 2. Count the current numbers
pos_count = len(filtered_df[filtered_df['Pneumonia'] == 1.0])
neg_count = len(filtered_df[filtered_df['Pneumonia'] == 0.0])

print(f"After filtering: Positive (Pneumonia) {pos_count} cases, Negative (Normal) {neg_count} cases.")

# 3. Save the final training list (includes labels for subsequent model training)
filtered_df.to_csv('../data/pneumonia_labeled.csv', index=False)

# 4. Construct PhysioNet download links
def build_url(row):
    subject_id = str(row['subject_id'])
    prefix = 'p' + subject_id[:2]  # Get the first two digits of subject_id as the prefix directory
    study_id = 's' + str(row['study_id'])
    dicom_id = row['dicom_id']
    # Construct standard path: .../pXX/pXXXXXX/sXXXXXX/dicom_id.jpg
    return f"https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/{prefix}/p{subject_id}/{study_id}/{dicom_id}.jpg"

# Apply the function to generate the URL column
filtered_df['url'] = filtered_df.apply(build_url, axis=1)

# 5. Save all links to a text file for wget
# Note: Do not include header (header=False) or index (index=False)
filtered_df['url'].to_csv('../data/url_list.txt', index=False, header=False)

print(f"-> Training list saved to: ../data/pneumonia_labeled_train.csv")
print(f"-> Download list generated: ../data/url_list.txt")

筛选后：阳性(Pneumonia) 1949 例，阴性(Normal) 3255 例。


In [ ]:
'''
After the code finishes running, you will get a url_list.txt file. 
Please open your Terminal, cd into the directory where this file is located, and run the following command to start the download:

wget -i url_list.txt --user yanweijin --ask-password -x -nH --cut-dirs=3

Password: 3TtN@vZ7nig5qEh
'''


In [7]:
!pip install tqdm requests